In [3]:
import pandas as pd

file_path = r"AISHE Final Report 2023-24.xlsx"

df = pd.read_excel(
    "C:/Users/abish/Documents/Datasets/AISHE Final Report 2023-24.xlsx",
    sheet_name="43ENRLT 6",
    header=None
)

print(df.shape)
print(df.head(10))

(226, 29)
          0          1                                                  2   \
0        NaN        NaN  Table 43. Estimated Student Enrolment Trends a...   
1  Sl. \nNo.  State/UTs                                              Ph.D.   
2        NaN        NaN                                               Male   
3          1          2                                                  3   
4        NaN  All India                                                NaN   
5        NaN    2023-24                                             172356   
6        NaN    2022-23                                             120981   
7        NaN    2021-22                                             113932   
8        NaN    2020-21                                             116764   
9        NaN    2019-20                                             111444   

       3       4        5       6      7               8               9   \
0     NaN     NaN      NaN     NaN    NaN             

In [4]:
print(df.shape)
print(df.iloc[:15, :10])

(226, 29)
            0              1  \
0         NaN            NaN   
1   Sl. \nNo.      State/UTs   
2         NaN            NaN   
3           1              2   
4         NaN      All India   
5         NaN        2023-24   
6         NaN        2022-23   
7         NaN        2021-22   
8         NaN        2020-21   
9         NaN        2019-20   
10          1  A & N Islands   
11        NaN        2023-24   
12        NaN        2022-23   
13        NaN        2021-22   
14        NaN        2020-21   

                                                    2       3       4  \
0   Table 43. Estimated Student Enrolment Trends a...     NaN     NaN   
1                                               Ph.D.     NaN     NaN   
2                                                Male  Female    Both   
3                                                   3       4       5   
4                                                 NaN     NaN     NaN   
5                                      

In [5]:
# Keep only the actual data rows
clean = df.iloc[5:].copy()

# Rename the columns
clean.columns = [
    "State_UT",
    "Year",
    "PhD_Male", "PhD_Female", "PhD_Total",
    "MPhil_Male", "MPhil_Female", "MPhil_Total",
    "PG_Male", "PG_Female", "PG_Total",
    "UG_Male", "UG_Female", "UG_Total",
    "PG_Diploma_Male", "PG_Diploma_Female", "PG_Diploma_Total",
    "Diploma_Male", "Diploma_Female", "Diploma_Total",
    "Certificate_Male", "Certificate_Female", "Certificate_Total",
    "Integrated_Male", "Integrated_Female", "Integrated_Total",
    "Grand_Total_Male", "Grand_Total_Female", "Grand_Total"
]

# Convert Year to text
clean["Year"] = clean["Year"].astype(str).str.strip()

# Remove rows without a year
clean = clean[clean["Year"].str.contains("-", na=False)]

# Convert numerical columns to numbers
numeric_cols = clean.columns[2:]

for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Reset index
clean = clean.reset_index(drop=True)

print(clean.shape)
print(clean.head())
print(clean.tail())

(185, 29)
  State_UT     Year  PhD_Male  PhD_Female  PhD_Total  MPhil_Male  \
0      NaN  2023-24  172356.0    171203.0   343559.0       668.0   
1      NaN  2022-23  120981.0    112441.0   233422.0      1270.0   
2      NaN  2021-22  113932.0     98636.0   212568.0      3393.0   
3      NaN  2020-21  116764.0     95088.0   211852.0      6345.0   
4      NaN  2019-20  111444.0     91106.0   202550.0      9043.0   

   MPhil_Female  MPhil_Total       PG_Male     PG_Female  ...  Diploma_Total  \
0        1418.0       2086.0  2.532582e+06  3.253270e+06  ...      3315728.0   
1        1964.0       3234.0  2.511463e+06  3.202950e+06  ...      3102075.0   
2        6127.0       9520.0  2.325040e+06  2.892713e+06  ...      2916445.0   
3       10399.0      16744.0  2.053794e+06  2.662855e+06  ...      2979320.0   
4       14891.0      23934.0  1.860163e+06  2.452372e+06  ...      2672562.0   

   Certificate_Male  Certificate_Female  Certificate_Total  Integrated_Male  \
0           97842.0  

In [6]:
# Fill State/UT names downward
clean["State_UT"] = clean["State_UT"].ffill()

# Check the result
print(clean[["State_UT", "Year", "Grand_Total"]].head(15))

   State_UT     Year   Grand_Total
0       NaN  2023-24  4.500112e+07
1       NaN  2022-23  4.463226e+07
2       NaN  2021-22  4.326818e+07
3       NaN  2020-21  4.138071e+07
4       NaN  2019-20  3.853636e+07
5       NaN  2023-24  7.128000e+03
6       NaN  2022-23  7.441000e+03
7       NaN  2021-22  1.142700e+04
8       NaN  2020-21  1.196500e+04
9       NaN  2019-20  1.013100e+04
10      NaN  2023-24  1.804368e+06
11      NaN  2022-23  1.840345e+06
12      NaN  2021-22  1.929159e+06
13      NaN  2020-21  1.987618e+06
14      NaN  2019-20  1.897149e+06


In [8]:
# Start fresh from the original dataframe
clean = df.iloc[5:].copy()

# The first column is Sl.No., second is State/UT
# Fill State/UT names from the original dataframe
state_names = df[1].where(
    ~df[1].astype(str).str.match(r"^\d{4}-\d{2}$", na=False)
).ffill()

# Keep only year rows
year_mask = df[1].astype(str).str.match(r"^\d{4}-\d{2}$", na=False)

clean = df.loc[year_mask].copy()

# Add the correct State/UT
clean["State_UT"] = state_names.loc[year_mask].values

# Remove Sl.No. and original State/UT columns
clean = clean.drop(columns=[0, 1])

# Put State_UT and Year first
clean = clean.rename(columns={2: "PhD_Male"})
clean.insert(0, "State_UT", clean.pop("State_UT"))
clean.insert(1, "Year", df.loc[year_mask, 1].values)

# Rename remaining columns
clean.columns = [
    "State_UT", "Year",
    "PhD_Male", "PhD_Female", "PhD_Total",
    "MPhil_Male", "MPhil_Female", "MPhil_Total",
    "PG_Male", "PG_Female", "PG_Total",
    "UG_Male", "UG_Female", "UG_Total",
    "PG_Diploma_Male", "PG_Diploma_Female", "PG_Diploma_Total",
    "Diploma_Male", "Diploma_Female", "Diploma_Total",
    "Certificate_Male", "Certificate_Female", "Certificate_Total",
    "Integrated_Male", "Integrated_Female", "Integrated_Total",
    "Grand_Total_Male", "Grand_Total_Female", "Grand_Total"
]

# Convert enrollment columns to numeric
for col in clean.columns[2:]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean = clean.reset_index(drop=True)

print(clean.shape)
print(clean[["State_UT", "Year", "Grand_Total"]].head(15))

(185, 29)
          State_UT     Year   Grand_Total
0        All India  2023-24  4.500112e+07
1        All India  2022-23  4.463226e+07
2        All India  2021-22  4.326818e+07
3        All India  2020-21  4.138071e+07
4        All India  2019-20  3.853636e+07
5    A & N Islands  2023-24  7.128000e+03
6    A & N Islands  2022-23  7.441000e+03
7    A & N Islands  2021-22  1.142700e+04
8    A & N Islands  2020-21  1.196500e+04
9    A & N Islands  2019-20  1.013100e+04
10  Andhra Pradesh  2023-24  1.804368e+06
11  Andhra Pradesh  2022-23  1.840345e+06
12  Andhra Pradesh  2021-22  1.929159e+06
13  Andhra Pradesh  2020-21  1.987618e+06
14  Andhra Pradesh  2019-20  1.897149e+06


In [9]:
print("Missing values:")
print(clean.isnull().sum())

print("\nNumber of States/UTs:", clean["State_UT"].nunique())

print("\nYears:")
print(clean["Year"].unique())

print("\nDuplicate rows:", clean.duplicated().sum())

Missing values:
State_UT               0
Year                   0
PhD_Male               8
PhD_Female             7
PhD_Total              7
MPhil_Male            37
MPhil_Female          36
MPhil_Total           33
PG_Male                3
PG_Female              3
PG_Total               3
UG_Male                0
UG_Female              0
UG_Total               0
PG_Diploma_Male        7
PG_Diploma_Female      7
PG_Diploma_Total       7
Diploma_Male           1
Diploma_Female         1
Diploma_Total          1
Certificate_Male       9
Certificate_Female    10
Certificate_Total      9
Integrated_Male       12
Integrated_Female     12
Integrated_Total      12
Grand_Total_Male       0
Grand_Total_Female     0
Grand_Total            0
dtype: int64

Number of States/UTs: 37

Years:
<StringArray>
['2023-24', '2022-23', '2021-22', '2020-21', '2019-20']
Length: 5, dtype: str

Duplicate rows: 0


In [10]:
clean.to_csv("AISHE_Enrollment_Trend_2019_2024.csv", index=False)

print("File saved successfully!")
print(clean.shape)
print(clean[["State_UT", "Year", "UG_Total", "PG_Total", "Grand_Total"]].head(10))

File saved successfully!
(185, 29)
        State_UT     Year      UG_Total      PG_Total   Grand_Total
0      All India  2023-24  3.458227e+07  5.785852e+06  4.500112e+07
1      All India  2022-23  3.467559e+07  5.714414e+06  4.463226e+07
2      All India  2021-22  3.413923e+07  5.217753e+06  4.326818e+07
3      All India  2020-21  3.265751e+07  4.716649e+06  4.138071e+07
4      All India  2019-20  3.064729e+07  4.312535e+06  3.853636e+07
5  A & N Islands  2023-24  4.913000e+03  1.305000e+03  7.128000e+03
6  A & N Islands  2022-23  5.125000e+03  1.285000e+03  7.441000e+03
7  A & N Islands  2021-22  8.251000e+03  1.517000e+03  1.142700e+04
8  A & N Islands  2020-21  8.727000e+03  1.561000e+03  1.196500e+04
9  A & N Islands  2019-20  7.456000e+03  1.540000e+03  1.013100e+04


In [13]:
ug_disc = pd.read_excel(
    "C:/Users/abish/Documents/Datasets/AISHE Final Report 2023-24.xlsx",
    sheet_name="12UGDisc ",
    header=None
)

print(ug_disc.shape)
print(ug_disc.iloc[:15, :10])

(131, 6)
                                                    0  \
0   Table 12.Undergraduate Enrolment Analysis by M...   
1                                            Sl. No.    
2                                                   1   
3                                                   1   
4                                                   2   
5                                                   3   
6                                                   4   
7                                                   5   
8                                                   6   
9                                                   7   
10                                                  8   
11                                                  9   
12                                                 10   
13                                                 11   
14                                                 12   

                                          1    2        3        4         5  

In [14]:
# Remove the report/header rows
ug_clean = ug_disc.iloc[3:].copy()

# Keep only the required columns
ug_clean = ug_clean.iloc[:, [1, 3, 4, 5]]

# Rename columns
ug_clean.columns = ["Discipline", "Male", "Female", "Total"]

# Remove blank disciplines
ug_clean = ug_clean.dropna(subset=["Discipline"])

# Convert enrollment values to numbers
for col in ["Male", "Female", "Total"]:
    ug_clean[col] = pd.to_numeric(ug_clean[col], errors="coerce")

# Reset index
ug_clean = ug_clean.reset_index(drop=True)

print(ug_clean.shape)
print(ug_clean.head(15))

print(ug_clean["Discipline"].tolist())

(63, 4)
                                 Discipline       Male   Female     Total
0                               Agriculture   208621.0    93278    301899
1                     Allied Health Science     2665.0     2943      5608
2        Animal Husbandry and Dairy Science       40.0       39        79
3                                   Apparel       27.0       48        75
4              Architecture and Engineering        1.0        1         2
5                              Area Studies    18874.0    18984     37858
6                                      Arts  5027958.0  5874503  10902461
7   Banking, Financial Services & Insurance      649.0      573      1222
8                                  Commerce  2123388.0  1957083   4080471
9                               Criminology      909.0      448      1357
10           Criminology & Forensic Science     1830.0     2145      3975
11                            Culinary Arts      156.0       29       185
12                         Cul

In [15]:
ug_clean = ug_clean[
    ~ug_clean["Discipline"].str.contains(" Total$", na=False)
].copy()

ug_clean = ug_clean.reset_index(drop=True)

print(ug_clean.shape)
print(ug_clean["Discipline"].tolist())

(61, 4)
['Agriculture', 'Allied Health Science', 'Animal Husbandry and Dairy Science', 'Apparel', 'Architecture and Engineering', 'Area Studies', 'Arts', 'Banking, Financial Services & Insurance', 'Commerce', 'Criminology', 'Criminology & Forensic Science', 'Culinary Arts', 'Cultural Studies', 'Defence Studies', 'Design', 'Education', 'Engineering & Technology', 'Engineering Science', 'Fashion & Apparel Design', 'Fashion Technology', 'Fine Arts', 'Fisheries Science', 'Food Processing', 'Footwear  Design', 'Footwear Technology', 'Foreign Language', 'Forest Management', 'Gandhian Studies', 'Health Care Profession (HCP)', 'Home Science', 'Hospitality and Tourism', 'Indian Language', 'Information Technology', 'IT & Computer', 'Journalism & Mass Communication', 'Law', 'Library & Information Science', 'Life Science & Health Care', 'Linguistics', 'Logistics', 'Management', 'Management & Entrepreneurship', 'Marine Science / Oceanography', 'Media & Entertainment', 'Medical Science', 'Oriental L

In [16]:
ug_clean.to_csv("AISHE_UG_Discipline_Enrollment_2023_24.csv", index=False)

print("UG discipline dataset saved successfully!")

UG discipline dataset saved successfully!


In [17]:
pg_disc = pd.read_excel(
    "C:/Users/abish/Documents/Datasets/AISHE Final Report 2023-24.xlsx",
    sheet_name="13PGDisc",
    header=None
)

print(pg_disc.shape)
print(pg_disc.iloc[:15, :10])

(298, 12)
             0                                                  1  \
0          NaN  Table 13.Enrolment Analysis by Major Disciplin...   
1   Sl. \nNo.                                          Discipline   
2          NaN                                                NaN   
3            1                                                  2   
4            1                                        Agriculture   
5            2                                                NaN   
6            3                                                NaN   
7            4                                                NaN   
8            5                                  Agriculture Total   
9            6                                            Apparel   
10           7                                      Apparel Total   
11           8                                       Area Studies   
12           9                                                NaN   
13          10          

In [18]:
pg_clean = pg_disc.iloc[4:].copy()

# Keep Discipline + the 9 enrollment columns
pg_clean = pg_clean.iloc[:, [1, 3, 4, 5, 6, 7, 8, 9, 10, 11]]

pg_clean.columns = [
    "Discipline",
    "PhD_Male", "PhD_Female", "PhD_Total",
    "MPhil_Male", "MPhil_Female", "MPhil_Total",
    "PG_Male", "PG_Female", "PG_Total"
]

# Remove rows where discipline is blank
pg_clean = pg_clean.dropna(subset=["Discipline"])

# Convert enrollment values to numeric
for col in pg_clean.columns[1:]:
    pg_clean[col] = pd.to_numeric(pg_clean[col], errors="coerce")

# Remove "Total" rows to avoid double counting
pg_clean = pg_clean[
    ~pg_clean["Discipline"].str.contains(" Total$", na=False)
].copy()

pg_clean = pg_clean.reset_index(drop=True)

print(pg_clean.shape)
print(pg_clean.head(15))

(51, 10)
                                 Discipline  PhD_Male  PhD_Female  PhD_Total  \
0                               Agriculture    4816.0      4366.0     9182.0   
1                                   Apparel       NaN         NaN        NaN   
2                              Area Studies    3711.0      4795.0     8506.0   
3                                      Arts       NaN         NaN        NaN   
4   Banking, Financial Services & Insurance       NaN         NaN        NaN   
5                                  Commerce       NaN         NaN        NaN   
6                               Criminology       NaN         NaN        NaN   
7            Criminology & Forensic Science      35.0        74.0      109.0   
8                          Cultural Studies     307.0       395.0      702.0   
9                            Cyber Security       NaN         NaN        NaN   
10                          Defence Studies     157.0        52.0      209.0   
11                             

In [19]:
pg_clean.to_csv("AISHE_PG_PhD_Discipline_Enrollment_2023_24.csv", index=False)

print("PG/PhD discipline dataset saved successfully!")

PG/PhD discipline dataset saved successfully!


In [20]:
programme = pd.read_excel(
    "C:/Users/abish/Documents/Datasets/AISHE Final Report 2023-24.xlsx",
    sheet_name="11Programme",
    header=None
)

print(programme.shape)
print(programme.iloc[:15, :10])

(268, 14)
                                                    0  \
0   Table 11. Program-Wise Enrolment across differ...   
1                                             Sl. No.   
2                                                 NaN   
3                                                   1   
4                                                   1   
5                                                   2   
6                                                   3   
7                                                   4   
8                                                   5   
9                                                   6   
10                                                  7   
11                                                  8   
12                                                  9   
13                                                 10   
14                                                 11   

                                                    1        2        3  \
0 

In [21]:
programme_clean = programme.iloc[4:].copy()

# Keep programme name + ALL enrollment columns
programme_clean = programme_clean.iloc[:, [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]]

programme_clean.columns = [
    "Programme",
    "All_Male", "All_Female", "All_Total",
    "Regular_Male", "Regular_Female", "Regular_Total",
    "Distance_Male", "Distance_Female", "Distance_Total",
    "Online_Male", "Online_Female", "Online_Total"
]

# Remove blank programme rows
programme_clean = programme_clean.dropna(subset=["Programme"])

# Convert enrollment columns to numeric
for col in programme_clean.columns[1:]:
    programme_clean[col] = pd.to_numeric(
        programme_clean[col], errors="coerce"
    )

# Reset index
programme_clean = programme_clean.reset_index(drop=True)

print(programme_clean.shape)
print(programme_clean.head(15))

(263, 13)
                                            Programme   All_Male  All_Female  \
0                    A.N.M.-Auxiliary Nurse & Midwife     1856.0    109356.0   
1                                             Acharya    12485.0      6194.0   
2                                     Alankar-Alankar       46.0         NaN   
3                                       Art Education      361.0       233.0   
4                Ayurveda Vachaspati-Ph.D in Ayurveda      544.0       679.0   
5                       Ayurvedacharya-Ayurvedacharya     7630.0     10963.0   
6   B.A. B.Ed.-Bachelor of Arts, Bachelor of Educa...    39248.0     57563.0   
7   B.A. L.L.B.-Bachelor of Arts, Bachelor of Law ...   110709.0     89146.0   
8           B.A. M.A.-Bachelor of Arts,Master of Arts     1100.0      1028.0   
9                B.A.(Hons)-Bachelor of Arts (Honors)   904187.0   1254235.0   
10                              B.A.-Bachelor of Arts  5042329.0   5893877.0   
11              B.A.M.-Bachelo

In [22]:
programme_clean.to_csv(
    "AISHE_Programme_Enrollment_2023_24.csv",
    index=False
)

print("Programme dataset saved successfully!")

Programme dataset saved successfully!


In [23]:
print(programme_clean["Programme"].tolist())

['A.N.M.-Auxiliary Nurse & Midwife', 'Acharya', 'Alankar-Alankar', 'Art Education', 'Ayurveda Vachaspati-Ph.D in Ayurveda', 'Ayurvedacharya-Ayurvedacharya', 'B.A. B.Ed.-Bachelor of Arts, Bachelor of Education', 'B.A. L.L.B.-Bachelor of Arts, Bachelor of Law or Laws', 'B.A. M.A.-Bachelor of Arts,Master of Arts', 'B.A.(Hons)-Bachelor of Arts (Honors)', 'B.A.-Bachelor of Arts', 'B.A.M.-Bachelor of Ayurvedic Medicine', 'B.A.M.S.-Bachelor of Ayurvedic Medicine & Surgery', 'B.A.S.L.P.-Bachelor of Audiology and Speech Language Pathology', 'B.Agri.-Bachelor of Agriculture', 'B.Arch-Bachelor of Architecture', 'B.B.A.(Hons)- Bachelor of Business Administration (Honors) ', 'B.B.A.-Bachelor of Business Administration', 'B.B.A-D&I-Bachelor of Business Administration in Design and Innovation', 'B.B.A-L.L.B(Hons)', 'B.B.A-L.L.B.-Bachelor of Business Administration, Bachelor of Law or Laws', 'B.B.M.-Bachelor of Business Management', 'B.B.S.-Bachelor of Business Studies', 'B.C.A. .(Hons)-Bachelor of Co

In [24]:
import os

print(os.getcwd())

c:\Users\abish\AppData\Local\Programs\Microsoft VS Code


In [25]:
import os

print(os.listdir())

['a5b5009513', 'AISHE_Enrollment_Trend_2019_2024.csv', 'AISHE_PG_PhD_Discipline_Enrollment_2023_24.csv', 'AISHE_Programme_Enrollment_2023_24.csv', 'AISHE_UG_Discipline_Enrollment_2023_24.csv', 'bin', 'Code.exe', 'Code.VisualElementsManifest.xml', 'unins000.dat', 'unins000.exe', 'unins000.msg']
